In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings("ignore")

In [2]:
from sklearn.datasets import make_classification
import numpy as np

X, y = make_classification(
    n_samples=400,           # 400 total samples
    n_features=12,           # 12 features
    n_informative=6,         # 6 features actually matter
    n_redundant=2,           # 2 features are copies
    n_classes=3,             # ← 3 CLASSES (multiclass!)
    n_clusters_per_class=2,  # Classes are scattered
    flip_y=0.08,             # 8% noise in labels
    random_state=42,
    class_sep=0.7            # Classes overlap a bit
)

print(f"Dataset shape: {X.shape}")
print(f"Class distribution:\n{np.bincount(y)}")
print(f"\nClasses: {np.unique(y)}")

Dataset shape: (400, 12)
Class distribution:
[137 132 131]

Classes: [0 1 2]


In [3]:
# column_names = []

# for i in range(12):
#     column_names.append(f'Feature_{i}')

# X_df = pd.DataFrame(X, columns=column_names)

import pandas as pd

# Convert to DataFrame
X_df = pd.DataFrame(X, columns=[f'Feature_{i}' for i in range(12)])
X_df['Target'] = y

print(X_df.head())
print(f"\nDataFrame shape: {X_df.shape}")
print(f"\nClass distribution:\n{X_df['Target'].value_counts().sort_index()}")
print(f"\nMissing values:\n{X_df.isnull().sum().sum()}")

   Feature_0  Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  \
0   0.626471   5.635698  -1.279031  -1.511694  -2.893711  -0.684456   
1   0.037870   0.600904  -0.712221   0.741232   0.125456   0.912346   
2   1.686950   2.119650  -1.422975  -0.948739  -0.954842  -2.223926   
3  -0.167179   0.530598  -1.097303   0.767983  -0.495990   1.445000   
4   0.781523   0.027293  -0.236555   1.775590   0.166822   0.073603   

   Feature_6  Feature_7  Feature_8  Feature_9  Feature_10  Feature_11  Target  
0   0.515294  -0.279993  -0.085769  -0.834282    0.271898   -0.575707       1  
1   1.186735  -0.881068  -1.764712   1.436335   -0.984234    1.162902       2  
2  -0.970124   1.036088   0.695643  -0.397558    2.707503    1.086290       1  
3  -0.588867   0.216776  -0.975348  -0.837262   -0.916921   -0.378210       0  
4   1.135620   0.932192   1.267549  -1.106294   -0.884357   -0.148733       1  

DataFrame shape: (400, 13)

Class distribution:
Target
0    137
1    132
2    131
Name: coun

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = X_df.drop('Target', axis=1)
y = X_df['Target']

# Train-test split with stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"\nTrain class distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nTest class distribution:\n{y_test.value_counts().sort_index()}")

Train shape: (320, 12)
Test shape: (80, 12)

Train class distribution:
Target
0    109
1    106
2    105
Name: count, dtype: int64

Test class distribution:
Target
0    28
1    26
2    26
Name: count, dtype: int64


In [6]:
""" 
🎯 Understanding Stratify with Your Data

Original distribution (before split):
Class 0: 137 out of 400 = 34.25%
Class 1: 132 out of 400 = 33.0%
Class 2: 131 out of 400 = 32.75%

Train distribution (320 samples):
Class 0: 109 out of 320 = 34.06%
Class 1: 106 out of 320 = 33.13%
Class 2: 105 out of 320 = 32.81%

Test distribution (80 samples):
Class 0: 28 out of 80 = 35.0%
Class 1: 26 out of 80 = 32.5%
Class 2: 26 out of 80 = 32.5%

"""

' \n🎯 Understanding Stratify with Your Data\n\nOriginal distribution (before split):\nClass 0: 137 out of 400 = 34.25%\nClass 1: 132 out of 400 = 33.0%\nClass 2: 131 out of 400 = 32.75%\n\nTrain distribution (320 samples):\nClass 0: 109 out of 320 = 34.06%\nClass 1: 106 out of 320 = 33.13%\nClass 2: 105 out of 320 = 32.81%\n\nTest distribution (80 samples):\nClass 0: 28 out of 80 = 35.0%\nClass 1: 26 out of 80 = 32.5%\nClass 2: 26 out of 80 = 32.5%\n\n'

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling done!")
print(f"Train scaled shape: {X_train_scaled.shape}")
print(f"Test scaled shape: {X_test_scaled.shape}")

Scaling done!
Train scaled shape: (320, 12)
Test scaled shape: (80, 12)


In [11]:
# training 
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    #multi_class="multinomial"
)

model.fit(X_train_scaled, y_train)

print(f"Coefficients shape : {model.coef_.shape}")
print(f"Intercept shape: {model.intercept_.shape}")

Coefficients shape : (3, 12)
Intercept shape: (3,)


12

In [18]:
# Create a DataFrame for better readability
coef_df = pd.DataFrame(
    model.coef_,
    columns=X.columns,
    index=['Class 0', 'Class 1', 'Class 2']
)

print("Coefficients for all features:\n")
print(coef_df.round(4))



Coefficients for all features:

         Feature_0  Feature_1  Feature_2  Feature_3  Feature_4  Feature_5  \
Class 0    -0.0217     0.1647    -0.0581     0.0471    -0.0773     -0.115   
Class 1     0.2647    -0.0324    -0.0144    -0.1355     0.3368     -0.009   
Class 2    -0.2430    -0.1324     0.0725     0.0885    -0.2595      0.124   

         Feature_6  Feature_7  Feature_8  Feature_9  Feature_10  Feature_11  
Class 0     0.1200    -0.0092    -0.1574     0.1175     -0.1534      0.3525  
Class 1    -0.0469    -0.0558     0.3816     0.0016      0.3443     -0.2925  
Class 2    -0.0731     0.0650    -0.2242    -0.1191     -0.1910     -0.0600  


In [19]:
print("\n\nIntercepts:")
for i, intercept in enumerate(model.intercept_):
    print(f"Class {i}: {intercept:.4f}")

print("\n")

class_names = ['Class 0', 'Class 1', 'Class 2']

for class_name, intercept in zip(class_names, model.intercept_):
    print(f"{class_name}: {intercept:.4f}")
    



Intercepts:
Class 0: 0.0499
Class 1: -0.0516
Class 2: 0.0017


Class 0: 0.0499
Class 1: -0.0516
Class 2: 0.0017


In [20]:
#prediction 
y_test_pred = model.predict(X_test_scaled)
y_test_pred_proba = model.predict_proba(X_test_scaled)

print(f"Test predictions shape: {y_test_pred.shape}")
print(f"Test probabilities shape: {y_test_pred_proba.shape}")


Test predictions shape: (80,)
Test probabilities shape: (80, 3)


In [22]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# accuracy 
test_accuracy = accuracy_score(y_test,y_test_pred)
print(f"Test Accuracy:  {test_accuracy:.4f}")

cm = confusion_matrix(y_test, y_test_pred)
print(f"\nConfusion Matrix:\n{cm}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Class 0', 'Class 1', 'Class 2']))

Test Accuracy:  0.5250

Confusion Matrix:
[[14  4 10]
 [ 2 18  6]
 [ 8  8 10]]

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.58      0.50      0.54        28
     Class 1       0.60      0.69      0.64        26
     Class 2       0.38      0.38      0.38        26

    accuracy                           0.53        80
   macro avg       0.52      0.53      0.52        80
weighted avg       0.52      0.53      0.52        80



In [ ]:
""" 
Let's start with the Confusion Matrix:
              Predicted
            Class 0  Class 1  Class 2
Actual 0      14      4       10
Actual 1       2     18        6
Actual 2       8      8       10

💡 How to Read This (Simply)
Row 0 (Actual Class 0, 28 samples):

14 correctly predicted as class 0 ✅
4 wrongly predicted as class 1 ❌
10 wrongly predicted as class 2 ❌

Row 1 (Actual Class 1, 26 samples):

2 wrongly predicted as class 0 ❌
18 correctly predicted as class 1 ✅
6 wrongly predicted as class 2 ❌

Row 2 (Actual Class 2, 26 samples):

8 wrongly predicted as class 0 ❌
8 wrongly predicted as class 1 ❌
10 correctly predicted as class 2 ✅
___________________________________________________________________________________________

From your classification report:
           precision    recall  f1-score
Class 0       0.58      0.50      0.54
Class 1       0.60      0.69      0.64
Class 2       0.38      0.38      0.38
Let me explain Precision for Class 2 first (since it's the worst):
Precision = 0.38 for Class 2
Question: "Of all samples the model predicted as Class 2, how many were actually Class 2?"
From confusion matrix, how many did model predict as Class 2?
Row 0, Col 2: 10 predicted as class 2 (but actually class 0)
Row 1, Col 2: 6 predicted as class 2 (but actually class 1)
Row 2, Col 2: 10 predicted as class 2 (actually class 2) ✓
Total: 10 + 6 + 10 = 26 predicted as class 2
Only 10 were correct

Precision = 10/26 = 0.38 (38%)
______________________________________________________________________________________________

✅ Now the Next Metric: Recall
From your report:
         recall
Class 0   0.50
Class 1   0.69
Class 2   0.38
Recall = Out of ACTUAL samples, how many did we catch?
For Class 2:

Actual Class 2 samples: 26
Correctly caught: 10
Recall = 10/26 = 0.38

Meaning: "Of the 26 actual Class 2 samples, we only caught 38%."
"""